In [1]:
!pip3 install boto3
!pip3 install pandas

In [2]:
import boto3
import os
import re
import unicodedata
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, lower, regexp_replace, when

# =========================
# CONFIG S3
# =========================

s3 = boto3.client("s3")

bucket_raw = "last-mile-optimization-raw-gabriel"
bucket_trusted = "last-mile-optimization-trusted-gabriel"

input_key = "previsao_15_dias.csv"

local_input = "/tmp/previsao.csv"
local_output_texto = "/tmp/output_texto"
local_output_numero = "/tmp/output_numero"

# =========================
# DOWNLOAD DO S3
# =========================

s3.download_file(bucket_raw, input_key, local_input)

# =========================
# SPARK
# =========================

spark = SparkSession.builder \
    .appName("etl_weather_forecast") \
    .getOrCreate()

df = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .csv(local_input)

# =========================
# FUNÇÃO SNAKE_CASE
# =========================

def to_snake_case(text):
    if text is None:
        return None

    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)

    return text

# =========================
# TRADUÇÃO DAS COLUNAS
# =========================

mapeamento_colunas = {
    "Data": "date",
    "Temperatura Máxima": "max_temperature",
    "Temperatura Mínima": "min_temperature",
    "Chuva (mm)": "rain_mm",
    "Código do Clima": "weather_code"
}

for coluna_antiga, coluna_nova in mapeamento_colunas.items():
    if coluna_antiga in df.columns:
        df = df.withColumnRenamed(coluna_antiga, coluna_nova)

# =========================
# PADRONIZAÇÃO NOMES
# =========================

df = df.toDF(*[to_snake_case(c) for c in df.columns])

# =========================
# PADRONIZAÇÃO STRINGS
# =========================

for campo, tipo in df.dtypes:
    if tipo == "string":
        df = df.withColumn(campo, trim(lower(col(campo))))
        df = df.withColumn(campo, regexp_replace(col(campo), r"\s+", "_"))

# =========================
# TRATAMENTO NULOS
# =========================

for campo in df.columns:
    df = df.withColumn(
        campo,
        when(trim(col(campo)) == "", None).otherwise(col(campo))
    )

# =========================
# CRIAÇÃO DESCRIÇÃO
# =========================

df = df.withColumn(
    "weather_description",

    when(col("weather_code") == "0", "clear_sky")
    .when(col("weather_code") == "1", "mainly_clear")
    .when(col("weather_code") == "2", "partly_cloudy")
    .when(col("weather_code") == "3", "overcast")
    .when(col("weather_code") == "45", "fog")
    .when(col("weather_code") == "48", "depositing_rime_fog")
    .when(col("weather_code") == "51", "light_drizzle")
    .when(col("weather_code") == "53", "moderate_drizzle")
    .when(col("weather_code") == "55", "dense_drizzle")
    .when(col("weather_code") == "61", "light_rain")
    .when(col("weather_code") == "63", "moderate_rain")
    .when(col("weather_code") == "65", "heavy_rain")
    .when(col("weather_code") == "80", "light_rain_showers")
    .when(col("weather_code") == "81", "moderate_rain_showers")
    .when(col("weather_code") == "82", "violent_rain_showers")
    .when(col("weather_code") == "95", "thunderstorm")
    .otherwise("unknown")
)

# =========================
# REMOVER DUPLICADOS
# =========================

df = df.dropDuplicates()

# =========================
# DATASETS
# =========================

df_texto = df
df_numero = df.drop("weather_description")

# =========================
# SALVAR LOCAL (SPARK)
# =========================

df_texto.coalesce(1).write.mode("overwrite").option("header", "true").csv(local_output_texto)
df_numero.coalesce(1).write.mode("overwrite").option("header", "true").csv(local_output_numero)

# =========================
# UPLOAD PARA S3
# =========================

def upload_folder(local_path, bucket, s3_path):
    for file in os.listdir(local_path):
        if file.startswith("part-"):
            s3.upload_file(
                os.path.join(local_path, file),
                bucket,
                f"{s3_path}/{file}"
            )

upload_folder(local_output_texto, bucket_trusted, "weather/texto")
upload_folder(local_output_numero, bucket_trusted, "weather/numero")

print("ETL completed successfully!")
print("Data uploaded to S3 (trusted)")

root
 |-- date: string (nullable = true)
 |-- max_temperature: string (nullable = true)
 |-- min_temperature: string (nullable = true)
 |-- rain_mm: string (nullable = true)
 |-- weather_code: string (nullable = true)
 |-- weather_description: string (nullable = false)

+----------+---------------+---------------+-------+------------+----------------------------+
|date      |max_temperature|min_temperature|rain_mm|weather_code|weather_description         |
+----------+---------------+---------------+-------+------------+----------------------------+
|2026-04-08|30.0           |19.5           |1.7    |51          |light_drizzle               |
|2026-04-07|28.2           |19.0           |0.6    |3           |overcast                    |
|2026-04-04|26.6           |19.8           |1.8    |3           |overcast                    |
|2026-04-01|25.4           |18.2           |7.3    |96          |thunderstorm_with_light_hail|
|2026-04-11|24.4           |19.2           |1.3    |51          